<div style="display:flex;align-items:center;gap:18px;padding:14px 0 18px 0;border-bottom:3px solid #0a4dcc;">
<img src="../assets/logo_senati.png" width="210">
<div>
<h1 style="margin:0;color:#0a3f99;">Trabajo Final · Fundamentos y Algoritmia para Inteligencia Artificial</h1>
<p style="margin:5px 0 0 0;font-size:16px;"><strong>Ing. Gunar Castro Iñigo</strong></p>
</div>
</div>

## 2. Limpieza y preprocesamiento de los datos

Ya sabemos qué contiene nuestro archivo. Ahora vamos a prepararlo para que pueda ser utilizado por un modelo de Machine Learning.

En esta etapa realizaremos tres tareas principales:

- completar valores faltantes;
- normalizar las variables numéricas;
- convertir las variables categóricas a un formato que el modelo pueda interpretar.

La idea es que todo quede dentro de un **pipeline**, para que el procedimiento sea ordenado y reproducible.

In [1]:
import pandas as pd

# Volvemos a leer el dataset para que este cuaderno pueda ejecutarse de manera independiente.
df = pd.read_csv("../dataset/rendimiento_estudiantil_senati.csv")

# Revisamos nuevamente los valores faltantes antes de preparar los datos.
df.isnull().sum()[df.isnull().sum() > 0]

horas_estudio_semana      4
asistencia_pct            4
tareas_completadas_pct    4
uso_plataforma_horas      4
promedio_prev             4
dtype: int64

### Separamos la variable que queremos predecir

En nuestro caso queremos estimar `promedio_final`. Por eso la guardaremos en `y`. El resto de las columnas que utilizaremos para aprender se guardarán en `X`.

No usaremos `rendimiento_categoria` porque se obtiene a partir del promedio final y provocaría fuga de información.

In [2]:
# X contiene las variables de entrada.
X = df.drop(columns=["promedio_final", "rendimiento_categoria"])

# y contiene la variable objetivo.
y = df["promedio_final"]

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)

Dimensión de X: (260, 13)
Dimensión de y: (260,)


### Definimos las columnas numéricas y categóricas

Las variables numéricas se completarán con la **mediana** y después se normalizarán con `StandardScaler`.

Las variables categóricas se completarán con el valor más frecuente y se transformarán con `OneHotEncoder`.

In [3]:
columnas_numericas = [
    "edad", "horas_estudio_semana", "asistencia_pct",
    "tareas_completadas_pct", "participacion_pct",
    "uso_plataforma_horas", "tutorias_mes",
    "promedio_prev", "examen_parcial", "proyecto_final"
]

columnas_categoricas = ["genero", "internet_calidad"]

print("Variables numéricas:", len(columnas_numericas))
print("Variables categóricas:", len(columnas_categoricas))

Variables numéricas: 10
Variables categóricas: 2


In [4]:
# Importamos las herramientas de Scikit-learn que vamos a utilizar.
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Pipeline para las variables numéricas.
proceso_numerico = Pipeline(steps=[
    ("imputar_nulos", SimpleImputer(strategy="median")),
    ("normalizar", StandardScaler())
])

# Pipeline para las variables categóricas.
proceso_categorico = Pipeline(steps=[
    ("imputar_nulos", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore"))
])

# Unimos ambos procesos en un solo transformador.
preprocesador = ColumnTransformer(transformers=[
    ("numericas", proceso_numerico, columnas_numericas),
    ("categoricas", proceso_categorico, columnas_categoricas)
])

print("Preprocesador construido correctamente.")

Preprocesador construido correctamente.


### Separamos entrenamiento y prueba

No debemos entrenar y evaluar con los mismos registros. Por eso reservaremos el 20 % de los datos para comprobar después qué tan bien funciona el modelo con información que no utilizó durante el entrenamiento.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Registros para entrenamiento:", len(X_train))
print("Registros para prueba:", len(X_test))

Registros para entrenamiento: 208
Registros para prueba: 52


In [6]:
# Aplicamos el preprocesamiento a los datos de entrenamiento.
X_train_procesado = preprocesador.fit_transform(X_train)

# En los datos de prueba solamente transformamos; no volvemos a aprender parámetros.
X_test_procesado = preprocesador.transform(X_test)

print("Forma del conjunto de entrenamiento procesado:", X_train_procesado.shape)
print("Forma del conjunto de prueba procesado:", X_test_procesado.shape)

Forma del conjunto de entrenamiento procesado: (208, 15)
Forma del conjunto de prueba procesado: (52, 15)


### Cierre de esta etapa

Ya dejamos preparado el dataset para Machine Learning. Sin embargo, antes de entrenar el modelo vamos a desarrollar la parte matemática solicitada en el trabajo final.

**En el siguiente cuaderno trabajaremos con vectores, matrices, producto punto, norma y un sistema de ecuaciones lineales usando NumPy.**

---
**Ing. Gunar Castro Iñigo · SENATI**  
Este cuaderno forma parte de una secuencia de trabajo. Continúa con el siguiente archivo para mantener el orden del proyecto.